# **Step1: Data Ingestion & Validation**

In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Validation
import pandera as pa
from pandera import Column, DataFrameSchema, Check

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)


# Data Ingestion

In [2]:
# Load dataset
df = pd.read_csv("ESG_Data.csv")

# Preview data
df.head()


,Date,Year,Asset_ID,Asset_Type,Location,Operational_Status,Energy_Type,Consumption_Units,Emission_Type,Emissions_tCO2e,Target_Emissions_tCO2e,Reduction_Percentage_vs_BaseYear
0,2020-01-01,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2357.044000,Scope 1,0.645377,15004.345000,0.000000
1,2020-01-01,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2357.044000,Scope 2,0.530871,15004.345000,0.000000
2,2020-01-02,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2440.574000,Scope 1,0.688000,15004.345000,0.000000
3,2020-01-02,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2440.574000,Scope 2,0.572372,15004.345000,0.000000
4,2020-01-03,2020,A001,ServiceHub,England,Active,ELECTRICITY_KWH,2396.825000,Scope 1,0.715998,15004.345000,0.000000


# Initial Data Inspection

In [3]:
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())


Shape: (236736, 12)

Data Types:
 Date                                 object
Year                                  int64
Asset_ID                             object
Asset_Type                           object
Location                             object
Operational_Status                   object
Energy_Type                          object
Consumption_Units                   float64
Emission_Type                        object
Emissions_tCO2e                     float64
Target_Emissions_tCO2e              float64
Reduction_Percentage_vs_BaseYear    float64
dtype: object

Missing Values:
 Date                                0
Year                                0
Asset_ID                            0
Asset_Type                          0
Location                            0
Operational_Status                  0
Energy_Type                         0
Consumption_Units                   0
Emission_Type                       0
Emissions_tCO2e                     0
Target_Emissions_tCO2e    

# Data Cleaning & Standardization

In [4]:
# convert Date column properly
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [5]:
# Ensure Numeric Columns Are Correct Type
numeric_cols = [
    "Consumption_Units",
    "Emissions_tCO2e",
    "Target_Emissions_tCO2e",
    "Reduction_Percentage_vs_BaseYear"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


In [6]:
# Standardize Categorical Columns
categorical_cols = [
    "Asset_Type",
    "Location",
    "Operational_Status",
    "Energy_Type",
    "Emission_Type"
]

for col in categorical_cols:
    df[col] = df[col].str.strip().str.upper()


In [7]:
#                Remove Duplicates
# df = df.drop_duplicates()

#                  Drop Records with Critical Missing Values
# df = df.dropna(subset=["Date", "Emissions_tCO2e", "Consumption_Units"])


# Data Validation with Pandera

In [8]:
schema = DataFrameSchema({
    "Date": Column(pa.DateTime),
    "Asset_ID": Column(pa.String),
    "Asset_Type": Column(pa.String),
    "Location": Column(pa.String),
    "Operational_Status": Column(pa.String, Check.isin(["ACTIVE"])),
    "Energy_Type": Column(pa.String),
    "Emission_Type": Column(pa.String, Check.isin(["SCOPE 1", "SCOPE 2"])),
    "Consumption_Units": Column(pa.Float, Check.ge(0)),
    "Emissions_tCO2e": Column(pa.Float, Check.ge(0)),
    "Target_Emissions_tCO2e": Column(pa.Float, Check.ge(0)),
    "Reduction_Percentage_vs_BaseYear": Column(pa.Float, Check.ge(0)),
})

# Validate
df = schema.validate(df)

print("Data validation successful.")


/home/sundayofunmi/.pandas_venv/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:146: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


Data validation successful.


# Feature Engineering (Create Analysis-Ready Fields)

In [10]:
# Create Time-Based Features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter
df["YearMonth"] = df["Date"].dt.to_period("M")

# Create Scope-Specific Flags. i.e., "One-Hot Encoding" (or creating Binary Flags)
df["Is_Scope1"] = np.where(df["Emission_Type"] == "SCOPE 1", 1, 0)
df["Is_Scope2"] = np.where(df["Emission_Type"] == "SCOPE 2", 1, 0)


In [11]:
# Emission Intensity (It measures the "cleanliness" of the energy source or process.)
df["Emission_Intensity"] = df["Emissions_tCO2e"] / df["Consumption_Units"]


    Why Emission_Intensity is critical for our analysis
    Without this metric, the data can be misleading.

    The Problem: If a factory emits more CO2 this month, is it because they became less efficient, or just because they produced more goods?
    The Solution: Emission Intensity filters out the "size" of the operation. If the total emissions go up, but the Emission Intensity stays the same, the efficiency hasn't changed—it just means more energy is used.

# Create Curated Aggregated Tables

In [12]:
# 1. Monthly Emissions by Scope (For Forecasting)
monthly_scope = (
    df.groupby(["YearMonth", "Emission_Type"])["Emissions_tCO2e"]
      .sum()
      .reset_index()
)

monthly_scope["YearMonth"] = monthly_scope["YearMonth"].dt.to_timestamp()

monthly_scope.head()


,YearMonth,Emission_Type,Emissions_tCO2e
0,2020-01-01,SCOPE 1,1755.795813
1,2020-01-01,SCOPE 2,1908.584310
2,2020-02-01,SCOPE 1,1679.152641
3,2020-02-01,SCOPE 2,1792.904592
4,2020-03-01,SCOPE 1,1871.382396


In [13]:
# 2. Asset-Level Monthly Dataset
asset_monthly = (
    df.groupby(["Asset_ID", "YearMonth"])
      .agg({
          "Emissions_tCO2e": "sum",
          "Consumption_Units": "sum"
      })
      .reset_index()
)

asset_monthly["YearMonth"] = asset_monthly["YearMonth"].dt.to_timestamp()

asset_monthly.head()


,Asset_ID,YearMonth,Emissions_tCO2e,Consumption_Units
0,A001,2020-01-01,107.992008,258371.492000
1,A001,2020-02-01,103.836612,252351.442000
2,A001,2020-03-01,112.223889,274245.912000
3,A001,2020-04-01,112.890954,278310.858000
4,A001,2020-05-01,119.980866,299997.706000


In [14]:
# 3. Scope 1 vs Scope 2 Wide Format (Forecast-Ready)
scope_wide = (
    monthly_scope
        .pivot(index="YearMonth",
               columns="Emission_Type",
               values="Emissions_tCO2e")
        .reset_index()
)

scope_wide.columns.name = None

scope_wide.head()


,YearMonth,SCOPE 1,SCOPE 2
0,2020-01-01,1755.795813,1908.584310
1,2020-02-01,1679.152641,1792.904592
2,2020-03-01,1871.382396,1920.870573
3,2020-04-01,1887.235869,1870.897161
4,2020-05-01,1999.483530,1913.012361


In [15]:
# Save Curated Datasets
monthly_scope.to_csv("curated_monthly_scope.csv", index=False)
asset_monthly.to_csv("curated_asset_monthly.csv", index=False)
scope_wide.to_csv("curated_scope_wide.csv", index=False)

print("Curated datasets saved successfully.")


Curated datasets saved successfully.
